In [ ]:
# Sam Brown
# sam_brown@mines.edu
# June 6, 2025
# Goal: Does form factor relate at to slip size 

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import numpy as np 
import scipy
import matplotlib.pyplot as plt
import pandas as pd

import my_lib.funcs

# We will use 2011 data as many GZ stations are up during this time.
evts_path = "/Users/sambrown04/Documents/SURF/Events/2011_2011Events2stas"

In [ ]:
events_list = my_lib.funcs.load_evt(evts_path)

# Get start times
start_times = []
for event in events_list:
    start_times.append(event['time'][0])

# Define new dataframe 
n_df = pd.DataFrame(columns = ['Time', 'event_num', 'mins_since', 'avg_delta'])
n_df['Time'] = start_times
n_df['event_num'] = None
n_df['mins_since'] = None
n_df['avg_delta'] = None

n_df['event_num'] = n_df.reset_index().index # Keep events numbers for sorting

# Sort events to calculate time since last event.
n_df['Time'] = pd.to_datetime(n_df['Time'], errors='coerce')
n_df = n_df.sort_values('Time')

# Convert to minutes
n_df['mins_since'] = n_df['Time'].diff().dt.total_seconds() / 60

n_df = n_df.reset_index(drop = True) # Reset Row numbers

# Get average delta only for gz stations
feat = my_lib.funcs.extract_event_features(events_list)

# Get only rows that have gz stations
filt_events = []
for event in feat:
    gz_rows = event[event['station'].str.contains('gz')]

    filt_events.append(gz_rows if not gz_rows.empty else None)
# print(filt_events[0].head(15))

slip_averages = []
for gz_rows in filt_events:
    if gz_rows is None:
        slip_averages.append(np.nan)   # Placeholder for missing gz data
    else:
        slip_averages.append(gz_rows['total_delta'].mean())


# print(slip_averages)

for idx,row in n_df.iterrows():
    # print(idx, row)
    n_df.at[idx, 'avg_delta'] = slip_averages[n_df.at[idx, 'event_num']]

In [ ]:
n_df = n_df.dropna()


In [ ]:
# now we need to add a column for the category of form factor each event is in when the event happened
tide_time = my_lib.funcs.get_tide_data(events_list, 'gz07', days = 365, plot = True)

# print(tide_time.head())
# FORM FACTOR
reference_time = tide_time['time'].iloc[0]
seconds = [(date - reference_time).total_seconds() for date in tide_time['time']]

# print(tide_time['tide_height'].shape, tide_time['time'].shape)

tide = tide_time['tide_height']
dates_timeseries = tide_time['time']

spacing = 4  # Minutes
mean_days = 3
slide_days = 1
mean_units = int(mean_days * 24 * 60 / spacing)
slide_units = int(slide_days * 24 * 60 / spacing)

HR_TO_SEC = 3600
T_O1 = 25.81933871 * HR_TO_SEC
T_K1 = 23.93447213 * HR_TO_SEC
T_M2 = 12.4206012 * HR_TO_SEC
T_S2 = 12 * HR_TO_SEC

def sines(x, A1, phi1, A2, phi2):
    return A1 * np.sin(2 * np.pi * x / ((T_O1 + T_K1) / 2) + phi1) + A2 * np.sin(
        2 * np.pi * x / ((T_M2 + T_S2) / 2) + phi2
    )

form_factors = []
dates_form_factor = []
semidiurnal = []
diurnal = []

start = 0
end = mean_units
while end < len(seconds):
    seconds_tide = np.array(seconds[start:end], dtype=float)
    tide_window = np.array(tide[start:end], dtype=float)
    date_midpoint = dates_timeseries[(start + end) // 2]
    start += slide_units
    end += slide_units

    # Fit a sum of sines to the tide
    initial_guess = [50, 0, 50, 0]
    popt, pcov = scipy.optimize.curve_fit(sines, seconds_tide, tide_window, p0=initial_guess)

    # Extract fitted parameters
    Diurnal_fit, phi1_fit, SemiDiurnal_fit, phi2_fit = popt

    # Generate the fitted curve
    y_fit = sines(seconds_tide, Diurnal_fit, phi1_fit, SemiDiurnal_fit, phi2_fit)
    form_factor = np.abs(Diurnal_fit / SemiDiurnal_fit)
    semidiurnal.append((SemiDiurnal_fit))
    diurnal.append((Diurnal_fit))

    form_factors.append(form_factor)
    dates_form_factor.append(date_midpoint)

    


In [ ]:
df = pd.DataFrame(columns = ['dates_form_factor', 'form_factors'])
df['dates_form_factor'] = dates_form_factor
df['form_factors'] = form_factors

In [ ]:
n_df = n_df.sort_values('Time')

In [ ]:
n_df_matched = pd.merge_asof(n_df, df, left_on='Time', right_on='dates_form_factor', direction='nearest')
n_df['form_factors'] = n_df_matched['form_factors']

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Scatter 
scatter = ax.scatter(
    n_df['avg_delta'],
    n_df['mins_since'],
    c=n_df['form_factors'],   # color by form_factors
    cmap='viridis',
    s=50
)


cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Form Factors')


ax.set_xlabel('Average Delta Among Gz stations')
ax.set_ylabel('Minutes Since last Event')
ax.set_title('Event Scatter Plot colored by Form Factors')


ax.grid(True)

plt.show()